# mlgw_bbh vs bilby — golden BBH injection benchmark (GPU half)

Runs the **gwgpu_jax acceptance-walk sampler** with the **mlgw_bbh_jax**
waveform model (SEOBNRv5HM, (2,2)-mode-only, aligned-spin — the same model
used in the real-GW150914 notebooks) on a fresh, "golden" synthetic BBH
injection (see `examples/mlgw_injection_benchmark/bbh_case/README.md` in the
repo for the full injection design and matched-configuration table).

The frozen strain + PSD data is **not** regenerated on Colab — it was
generated once (CPU, deterministic seed) and must be uploaded to Google
Drive at `MyDrive/mlgw_injection_benchmark/bbh_case/data/` before running
this notebook, so that this GPU run and the companion CPU bilby run
(`run_bilby_bbh.py`, LAL `SEOBNRv5_ROM`) consume **byte-identical inputs**
(checksum-verified below) — regenerating independently on GPU risks tiny
floating-point drift from the NN forward pass across hardware, which would
break that guarantee.

Afterwards, copy this notebook's output back and overlay both posteriors
with `make_comparison_plot.py`.


## 1. Install GWgpu_jax with `[data,mlgw]` extras

In [ ]:
import importlib.util, os

RUNNING_ON_COLAB  = importlib.util.find_spec("google.colab") is not None
ALREADY_INSTALLED = importlib.util.find_spec("gwgpu_jax") is not None

if RUNNING_ON_COLAB and not ALREADY_INSTALLED:
    OWNER, REPO, BRANCH = "saulo-albuquerque-phys", "GWgpu-jax", "acceptance_walk_sampler"

    # Pin JAX to the 0.4.x line Colab's CUDA plugin still supports.
    !pip uninstall -y -q jax jaxlib jax-cuda12-plugin jax-cuda12-pjrt 2>/dev/null
    !pip install -q "jax[cuda12]==0.4.31" "jaxlib==0.4.31"

    # Public repo — no GitHub token needed.
    !pip install -q "gwgpu_jax[data,mlgw] @ git+https://github.com/{OWNER}/{REPO}.git@{BRANCH}"

    print("Installed. If JAX was re-pinned: Runtime -> Restart session, then re-run from here.")
else:
    print("gwgpu_jax already importable - skipping install.")


### 1b. Pull the vendored `mlgw_bbh_jax` sub-repo (Colab bandaid)

IMPORTANT: if this is a fresh install (previous cell actually installed
something), restart the runtime now (Runtime -> Restart session) and run
from this cell down.

`mlgw_bbh_jax` is shipped as a *git submodule* at
`<gwgpu_jax-install>/mlgw_jax/mlgw_bbh_jax/` — it is not bundled into the
`gwgpu_jax` package Colab installs from git, so the wheel is missing it. The
cell below clones it on the fly from the **`gwjax_compatible`** branch of
`adrianomascioli/MLGW-JAX`, which already contains the inference-compatibility
fixes (same recipe as the real-GW150914 notebook).

In [ ]:
import subprocess, sys, shutil
from pathlib import Path
import gwgpu_jax

GWJAX_ROOT = Path(gwgpu_jax.__file__).resolve().parent
DEST = GWJAX_ROOT / "mlgw_jax" / "mlgw_bbh_jax"

if not (DEST / "mlgw" / "GW_generator.py").exists():
    print(f"Cloning mlgw_bbh_jax into {DEST} ...")
    if DEST.exists():
        shutil.rmtree(DEST)
    subprocess.run(["git", "clone", "--depth", "1", "-b", "gwjax_compatible",
                    "https://github.com/adrianomascioli/MLGW-JAX.git", str(DEST)], check=True)
    print("Done.")
else:
    print("mlgw_bbh_jax already present — skipping clone.")


## 2. Fetch the external acceptance-walk kernel (Prathaban et al. 2025)

In [ ]:
import importlib.util, os, sys, subprocess

if importlib.util.find_spec("custom_kernels") is None:
    KDIR = "/content/blackjax_ns_gw" if os.path.isdir("/content") else "external/blackjax_ns_gw"
    if not os.path.isdir(f"{KDIR}/src/custom_kernels"):
        subprocess.run(["git", "clone", "--no-checkout", "--depth", "1", "--filter=blob:none",
                        "https://github.com/mrosep/blackjax_ns_gw.git", KDIR], check=True)
        subprocess.run(["git", "-C", KDIR, "sparse-checkout", "set", "src/custom_kernels"], check=True)
        subprocess.run(["git", "-C", KDIR, "checkout"], check=True)
    sys.path.insert(0, f"{KDIR}/src")
    os.environ["GWJAX_ACCEPTANCE_WALK_SRC"] = f"{KDIR}/src"
    import custom_kernels
    print("acceptance-walk kernel ready:", custom_kernels.__file__)
else:
    print("custom_kernels already importable.")


## 3. Frozen data — load from Google Drive and verify checksums

Upload the local `examples/mlgw_injection_benchmark/bbh_case/data/` folder
(generated by `generate_injection.py`) to your Google Drive at
`MyDrive/mlgw_injection_benchmark/bbh_case/data/` **before** running this
cell. The checksums below must match `data/metadata.json`'s `sha256_16` on
your machine — if they don't, re-upload (or re-run `generate_injection.py`
and re-upload — see the README for why regenerating on GPU is discouraged
for the frozen benchmark itself).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
from pathlib import Path

DRIVE_DATA = Path('/content/drive/MyDrive/mlgw_injection_benchmark/bbh_case/data')
DATA = Path('data')
if not DRIVE_DATA.is_dir():
    raise FileNotFoundError(
        f"{DRIVE_DATA} not found. Upload examples/mlgw_injection_benchmark/bbh_case/data/ "
        "to your Google Drive at this path first.")
shutil.copytree(DRIVE_DATA, DATA, dirs_exist_ok=True)
print(f"Copied frozen data from {DRIVE_DATA} to {DATA.resolve()}")


In [ ]:
import hashlib, json
import numpy as np

meta = json.load(open(DATA / "metadata.json"))
SHA = meta["sha256_16"]

def _sha(a) -> str:
    return hashlib.sha256(np.ascontiguousarray(a).tobytes()).hexdigest()[:16]

freqs_np = np.load(DATA / "frequency_array.npy")
for key, fname in [("frequency_array", "frequency_array.npy")] + [
        (f"{d}_{k}", f"{d}_{k}.npy") for d in meta["detectors"] for k in ("strain_fd", "psd")]:
    got = _sha(np.load(DATA / fname))
    if got != SHA[key]:
        raise RuntimeError(f"checksum mismatch for {fname}: {got} != {SHA[key]} "
                           "— data/ does not match the frozen injection metadata")
print("frozen mlgw_bbh injection verified against checksums.")
print("injection:", meta["injection"])
print("network optimal SNR of injected template:", meta["network_optimal_snr_of_injected_template"])


## 4. Build the gwjax network and the mlgw_bbh_jax waveform

In [ ]:
import time
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

import gwgpu_jax
from gwgpu_jax.gwgpu_jax_compatibility import gmst_from_gps

grid = gwgpu_jax.TimeFrequencyGrid(
    meta["duration"], meta["sampling_frequency"],
    f_min=meta["minimum_frequency"], f_max=meta["maximum_frequency"])
assert np.array_equal(np.asarray(grid.frequency_domain_array), freqs_np)

net = gwgpu_jax.Network.from_names(meta["detectors"], grid)
for ifo in net.interferometers:
    ifo.load_data(jnp.asarray(np.load(DATA / f"{ifo.name}_strain_fd.npy")), domain="fd")
    ifo.psd = jnp.asarray(np.load(DATA / f"{ifo.name}_psd.npy"))
    ifo._weight = None
net.gmst = gmst_from_gps(meta["trigger_time"])
print(net)


In [ ]:
!pip install -q -U pyOpenSSL cryptography

In [ ]:
# mlgw_bbh (SEOBNRv5HM, (2,2) mode) TD -> FD waveform — same recipe as
# run_gwjax_bbh.py / the real-GW150914 notebook.
from gwgpu_jax.mlgw_jax.mlgw_bbh_jax_waveform_generator import MLGWBBHGenerator

# Must equal the value the injection was generated with (merger at tc - final_time);
# frozen data made before Nov 2026 has no key and used 0.01 s (truncated ringdown).
MLGW_FINAL_TIME = float(meta.get("mlgw_final_time", 0.01))
print(f"mlgw_final_time = {MLGW_FINAL_TIME} s")
TUKEY_ALPHA_LEFT, TUKEY_ALPHA_RIGHT = 0.2, 0.001


def _tukey_asymmetric(n, alpha_left, alpha_right):
    x = jnp.linspace(0.0, 1.0, n)
    w = jnp.ones_like(x)
    left = 0.5 * (1.0 + jnp.cos(2.0 * jnp.pi * (x / alpha_left - 0.5)))
    right = 0.5 * (1.0 + jnp.cos(2.0 * jnp.pi * ((x - 1.0) / alpha_right + 0.5)))
    w = jnp.where(x < alpha_left / 2.0, left, w)
    w = jnp.where(x > 1.0 - alpha_right / 2.0, right, w)
    return w


def build_mlgw_bbh_waveform_fn(grid, modes=(2, 2),
                                final_time=MLGW_FINAL_TIME,
                                alpha_left=TUKEY_ALPHA_LEFT,
                                alpha_right=TUKEY_ALPHA_RIGHT):
    bbh = MLGWBBHGenerator(model=4)
    dt, N = grid.dt, grid.n_samples
    t_grid = jnp.arange(N) * dt - grid.duration + final_time
    window = _tukey_asymmetric(N, alpha_left, alpha_right)
    gen_wf = bbh.generate

    def waveform_fn(params, freqs):
        mc, q = params["mc"], params["q"]
        m1 = mc * (1.0 + q) ** (1.0 / 5.0) / q ** (3.0 / 5.0)
        m2 = q * m1
        theta = jnp.stack([
            m1, m2, params["chi_1"], params["chi_2"],
            params["distance"], params["inclination"], params["phi_c"],
        ])
        hp_td, hc_td = gen_wf(theta, t_grid, modes=modes)
        hp_fd = jnp.fft.rfft(hp_td * window) * dt
        hc_fd = jnp.fft.rfft(hc_td * window) * dt
        return hp_fd, -hc_fd

    return waveform_fn


t0 = time.perf_counter()
waveform_fn = build_mlgw_bbh_waveform_fn(grid, modes=(2, 2))
print(f"mlgw_bbh_jax (SEOBNRv5HM, model_4, (2,2) mode) loaded in {time.perf_counter()-t0:.1f} s")


## 5. Set up the acceptance-walk sampler

In [ ]:
TC_CENTER = meta["tc_in_segment"] + MLGW_FINAL_TIME   # mlgw tc that puts the merger on the trigger
inj = meta["injection"]
PARAM_BOUNDS = {
    "mc":          (inj["chirp_mass"] - 2.0, inj["chirp_mass"] + 2.0),
    "q":           (0.6, 1.0),
    "chi_1":       (-0.4, 0.4),
    "chi_2":       (-0.4, 0.4),
    "distance":    (50.0, 300.0),
    "inclination": (0.0, float(jnp.pi)),
    "ra":          (0.0, 2.0 * float(jnp.pi)),
    "dec":         (-float(jnp.pi) / 2, float(jnp.pi) / 2),
    "psi":         (0.0, float(jnp.pi)),
    "phi_c":       (0.0, 2.0 * float(jnp.pi)),
    "tc":          (TC_CENTER - 0.1, TC_CENTER + 0.1),
}
PRIORS = {
    "inclination": "sin",
    "dec":         "cos",
    "distance":    "volumetric",
}

sampler = gwgpu_jax.GWgpu_jaxAcceptanceWalkSampler(
    network=net, waveform_fn=waveform_fn,
    param_bounds=PARAM_BOUNDS, priors=PRIORS, gmst=None,
    periodic_params=("phi_c", "ra"),
)
print(f"sampling dimension : {len(sampler.param_bounds)}")
print(f"free parameters    : {list(sampler.param_bounds)}")


## 6. Run the sampler — the long step

`verbose=True` prints periodically. If you want reassurance mid-run, open a
new cell and run `!nvidia-smi` — GPU utilization near 100% confirms it's
actively sampling.

In [ ]:
NUM_LIVE   = 2000
N_TARGET   = 200
MAX_MCMC   = 1200
NUM_DELETE = 50
SEED       = 0
MAX_ITERS  = 10000

t0 = time.perf_counter()
result = sampler.run(
    rng_key               = jax.random.PRNGKey(SEED),
    num_live              = NUM_LIVE,
    n_target              = N_TARGET,
    max_mcmc              = MAX_MCMC,
    num_delete            = NUM_DELETE,
    max_iterations        = MAX_ITERS,
    log_dlogz_target      = -3,
    num_posterior_samples = 10000,
    verbose               = True,
)
elapsed = time.perf_counter() - t0
print(f"\nDone in {elapsed/3600:.2f} h. logZ = {result.logZ:.4f} +/- {result.logZ_err:.4f}   ESS = {result.ess:.1f}")


## 7. Save the posterior, zip, and copy back to Google Drive

In [ ]:
import json
import numpy as np
from pathlib import Path

noise_logl = float(sum(i.log_likelihood_data(jnp.zeros_like(i.strain_data_fd))
                       for i in net.interferometers))

OUTDIR = Path("gwjax_bbh_case")
OUTDIR.mkdir(parents=True, exist_ok=True)
post = {k: np.asarray(v) for k, v in result.posterior_samples.items()}
np.savez_compressed(OUTDIR / "posterior_samples.npz", **post)
json.dump(dict(
    logZ=float(result.logZ), logZ_err=float(result.logZ_err),
    log_noise_evidence=noise_logl,
    log_bayes_factor=float(result.logZ) - noise_logl,
    ess=float(result.ess), n_iterations=int(result.n_iterations),
    elapsed_s=elapsed, start_time=meta["start_time"],
    trigger_time=meta["trigger_time"], injection=meta["injection"],
    config=dict(nlive=NUM_LIVE, num_delete=NUM_DELETE, n_target=N_TARGET,
                max_mcmc=MAX_MCMC, log_dlogz_target=-3, seed=SEED,
                mlgw_final_time=MLGW_FINAL_TIME,
                waveform="mlgw_bbh_jax (SEOBNRv5HM, model_4, (2,2) mode)"),
), open(OUTDIR / "result.json", "w"), indent=2)

print(f"logZ             = {result.logZ:.4f} +/- {result.logZ_err:.4f}")
print(f"log_bayes_factor = {result.logZ - noise_logl:.4f}")
print(f"Saved {OUTDIR}/posterior_samples.npz and {OUTDIR}/result.json")

import shutil
shutil.make_archive(str(OUTDIR), "zip", root_dir=".", base_dir=str(OUTDIR))


In [ ]:
import shutil, os

dest = '/content/drive/MyDrive/mlgw_injection_benchmark/bbh_case'
os.makedirs(dest, exist_ok=True)
shutil.copytree('gwjax_bbh_case', f'{dest}/gwjax_bbh_case', dirs_exist_ok=True)
shutil.copy('gwjax_bbh_case.zip', dest)
print('Copied to Google Drive:', dest)
!ls -la {dest}/gwjax_bbh_case


## 8. Cross-check: ripplegw IMRPhenomXAS recovery (same pipeline, independent waveform)

A second, independent PE run on the **same frozen data**, **same gwjax
acceptance-walk sampler**, **same priors** as the `mlgw_bbh_jax` run above —
but recovering with **ripplegw's `IMRPhenomXAS`** instead of the NN
surrogate. ripplegw is independently validated to agree with LAL to ~1e-9
(`frozen_benchmark/README.md`), so this isolates variables cleanly:

* Any gap between **this** run and bilby's `SEOBNRv5_ROM` result (cluster
  side) is attributable to the small, already-characterized
  IMRPhenomXAS-vs-SEOBNRv5 model-family systematic (~0.6% mismatch at these
  parameters — see `generate_injection.py`'s triangulation check) — *not*
  to any gwjax-vs-bilby pipeline difference, since both this run and the
  `mlgw_bbh_jax` run above share the identical gwjax sampler.
* Any gap between **this** run and the `mlgw_bbh_jax` run above is
  attributable to the NN surrogate specifically, with the sampler held
  fixed.

In [ ]:
# ripplegw IMRPhenomXAS waveform_fn — (mc, q) -> (m1, m2) wrapper around the
# ripple builder, same pattern as frozen_benchmark/run_gwjax_frozen.py.
_base_wf_ripple = gwgpu_jax.build_ripplegw_waveform_fn(
    "IMRPhenomXAS", f_ref=meta["reference_frequency"])


def waveform_fn_ripple(params, freqs):
    mc, q = params["mc"], params["q"]
    m1 = mc * (1.0 + q) ** (1.0 / 5.0) / q ** (3.0 / 5.0)
    m2 = q * m1
    p = dict(params)
    p["m1"], p["m2"] = m1, m2
    return _base_wf_ripple(p, freqs)


print("ripplegw IMRPhenomXAS waveform_fn ready.")


In [ ]:
# Same PARAM_BOUNDS / PRIORS as the mlgw_bbh_jax run (cell under section 5) —
# a fair comparison needs identical priors, only the waveform_fn differs. The one
# exception is the tc window: ripple puts its merger AT tc, mlgw at tc - MLGW_FINAL_TIME,
# so ripple's window is centred on the physical merger (same physical prior).
PARAM_BOUNDS_RIPPLE = dict(PARAM_BOUNDS,
    tc=(meta["tc_in_segment"] - 0.1, meta["tc_in_segment"] + 0.1))
sampler_ripple = gwgpu_jax.GWgpu_jaxAcceptanceWalkSampler(
    network=net, waveform_fn=waveform_fn_ripple,
    param_bounds=PARAM_BOUNDS_RIPPLE, priors=PRIORS, gmst=None,
    periodic_params=("phi_c", "ra"),
)
print(f"sampling dimension : {len(sampler_ripple.param_bounds)}")
print(f"free parameters    : {list(sampler_ripple.param_bounds)}")

t0 = time.perf_counter()
result_ripple = sampler_ripple.run(
    rng_key               = jax.random.PRNGKey(SEED),
    num_live              = NUM_LIVE,
    n_target              = N_TARGET,
    max_mcmc              = MAX_MCMC,
    num_delete            = NUM_DELETE,
    max_iterations        = MAX_ITERS,
    log_dlogz_target      = -3,
    num_posterior_samples = 10000,
    verbose               = True,
)
elapsed_ripple = time.perf_counter() - t0
print(f"\nDone in {elapsed_ripple/3600:.2f} h. logZ = {result_ripple.logZ:.4f} +/- "
      f"{result_ripple.logZ_err:.4f}   ESS = {result_ripple.ess:.1f}")


In [ ]:
OUTDIR_RIPPLE = Path("gwjax_bbh_case_ripple")
OUTDIR_RIPPLE.mkdir(parents=True, exist_ok=True)
post_ripple = {k: np.asarray(v) for k, v in result_ripple.posterior_samples.items()}
np.savez_compressed(OUTDIR_RIPPLE / "posterior_samples.npz", **post_ripple)
json.dump(dict(
    logZ=float(result_ripple.logZ), logZ_err=float(result_ripple.logZ_err),
    log_noise_evidence=noise_logl,
    log_bayes_factor=float(result_ripple.logZ) - noise_logl,
    ess=float(result_ripple.ess), n_iterations=int(result_ripple.n_iterations),
    elapsed_s=elapsed_ripple, start_time=meta["start_time"],
    trigger_time=meta["trigger_time"], injection=meta["injection"],
    config=dict(nlive=NUM_LIVE, num_delete=NUM_DELETE, n_target=N_TARGET,
                max_mcmc=MAX_MCMC, log_dlogz_target=-3, seed=SEED,
                waveform="ripplegw IMRPhenomXAS"),
), open(OUTDIR_RIPPLE / "result.json", "w"), indent=2)

print(f"logZ             = {result_ripple.logZ:.4f} +/- {result_ripple.logZ_err:.4f}")
print(f"log_bayes_factor = {result_ripple.logZ - noise_logl:.4f}")
print(f"Saved {OUTDIR_RIPPLE}/posterior_samples.npz and {OUTDIR_RIPPLE}/result.json")

shutil.make_archive(str(OUTDIR_RIPPLE), "zip", root_dir=".", base_dir=str(OUTDIR_RIPPLE))

dest_ripple = '/content/drive/MyDrive/mlgw_injection_benchmark/bbh_case'
os.makedirs(dest_ripple, exist_ok=True)
shutil.copytree('gwjax_bbh_case_ripple', f'{dest_ripple}/gwjax_bbh_case_ripple', dirs_exist_ok=True)
shutil.copy('gwjax_bbh_case_ripple.zip', dest_ripple)
print('Copied to Google Drive:', dest_ripple)


## 9. Corner plot — the two gwjax runs (bilby not ready yet)

Overlays `mlgw_bbh_jax` (red) against ripplegw `IMRPhenomXAS` (green).
**Self-contained**: mounts Drive and loads both results straight from
`MyDrive/mlgw_injection_benchmark/bbh_case/`, so it works even in a fresh
kernel/runtime that never ran the sampler cells above — only needs the two
result folders to already be on Drive (copied there by sections 6/8's save
cells). This is a **surrogate-vs-ripple-only** comparison (same gwjax
pipeline on both sides) — run section 10 below once bilby's result is
copied over too, for the full 3-way including the LAL side.

In [ ]:
from google.colab import drive
import os
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

import json
from pathlib import Path
import numpy as np

DRIVE_BASE = Path('/content/drive/MyDrive/mlgw_injection_benchmark/bbh_case')
gdir = DRIVE_BASE / 'gwjax_bbh_case'
rdir = DRIVE_BASE / 'gwjax_bbh_case_ripple'
for d in (gdir, rdir):
    if not d.is_dir():
        raise FileNotFoundError(
            f"{d} not found on Drive — check section 6/8's save-and-copy cells "
            "actually ran (they run right after each sampler finishes, before "
            "this section) before the kernel died.")

gres = json.load(open(gdir / 'result.json'))
gpost = dict(np.load(gdir / 'posterior_samples.npz'))
rres = json.load(open(rdir / 'result.json'))
rpost = dict(np.load(rdir / 'posterior_samples.npz'))

# mlgw_bbh_jax's TD->FD recipe (zero-crossing time grid + rFFT's implicit
# "index 0 = time 0" framing) reads out tc about MLGW_FINAL_TIME = 10 ms
# HIGHER than the standard tc=0-at-projection convention ripple/bilby use,
# for the identical physical merger time — characterized exactly by
# generate_injection.py's ripplegw triangulation check (best dtc = -10.00 ms
# needed on the ripple side to match mlgw's injected signal). Rigid
# correction, not a fit, so the tc panel compares like with like.
# = the mlgw_final_time the run used (in result.json config since Nov 2026; older runs: 0.01 s)
TC_OFFSET_MLGW = float(gres.get("config", {}).get("mlgw_final_time", 0.01))
gpost = dict(gpost)
gpost["tc"] = gpost["tc"] - TC_OFFSET_MLGW

inj = gres["injection"]
start_time = gres["start_time"]
TC_CENTER = inj["geocent_time"] - start_time   # matches the tc_in_segment convention
print(f"loaded gwjax ({gpost['mc'].size} samples) and gwjax-ripple ({rpost['mc'].size} samples) from Drive")

!pip install -q corner
import corner
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

names_cmp = ["mc", "q", "distance", "dec", "ra", "inclination",
             "chi_1", "chi_2", "tc", "psi", "phi_c"]
LABELS_cmp = {
    "mc": r"$M_c$", "q": r"$q$", "chi_1": r"$\chi_1$", "chi_2": r"$\chi_2$",
    "distance": r"$d_L$", "inclination": r"$\theta_{JN}$", "ra": r"$\alpha$",
    "dec": r"$\delta$", "psi": r"$\psi$", "phi_c": r"$\phi_c$", "tc": r"$t_c$",
}
TRUTH_cmp = dict(mc=inj["chirp_mass"], q=inj["mass_ratio"], chi_1=inj["chi_1"],
                 chi_2=inj["chi_2"], distance=inj["luminosity_distance"],
                 inclination=inj["theta_jn"], ra=inj["ra"], dec=inj["dec"],
                 psi=inj["psi"], phi_c=inj["phase"],
                 # truth on THIS axis is the standard-convention tc — same
                 # TC_OFFSET_MLGW shift applied to gpost["tc"] above, since
                 # the injection was generated at mlgw tc=TC_CENTER.
                 # (new injections put the merger exactly on the trigger)
                 tc=TC_CENTER - (0.0 if "mlgw_final_time" in gres.get("config", {}) else TC_OFFSET_MLGW))

g_arr = np.column_stack([gpost[n] for n in names_cmp])
r_arr = np.column_stack([rpost[n] for n in names_cmp])
lo = np.minimum(g_arr.min(axis=0), r_arr.min(axis=0))
hi = np.maximum(g_arr.max(axis=0), r_arr.max(axis=0))
pad = 0.03 * (hi - lo)
ranges = list(zip(lo - pad, hi + pad))

ckw = dict(labels=[LABELS_cmp[n] for n in names_cmp], range=ranges,
           truths=[TRUTH_cmp[n] for n in names_cmp], truth_color="black",
           plot_datapoints=False, plot_density=False, levels=(0.393, 0.865, 0.989),
           smooth=0.9, bins=40)

fig = corner.corner(g_arr, color="#d8443f", hist_kwargs=dict(density=True, color="#d8443f"), **ckw)
corner.corner(r_arr, fig=fig, color="#3fa84b", hist_kwargs=dict(density=True, color="#3fa84b"), **ckw)
fig.legend(handles=[
    mlines.Line2D([], [], color="#d8443f", label="gwjax + acceptance-walk (mlgw_bbh_jax)"),
    mlines.Line2D([], [], color="#3fa84b", label="gwjax + acceptance-walk (ripplegw IMRPhenomXAS)"),
], loc="upper right", fontsize=14, frameon=False)
fig.savefig("bbh_case_gwjax_only.png", dpi=150, bbox_inches="tight")
plt.show()

drg = rres["logZ"] - gres["logZ"]
sig_rg = (rres["logZ_err"]**2 + gres["logZ_err"]**2) ** 0.5
print(f"mlgw_bbh_jax logZ          = {gres['logZ']:.4f} +/- {gres['logZ_err']:.4f}")
print(f"ripplegw IMRPhenomXAS logZ = {rres['logZ']:.4f} +/- {rres['logZ_err']:.4f}")
print(f"Delta logZ (ripple - mlgw) = {drg:+.4f}  ({drg/sig_rg:+.1f} sigma) — surrogate-vs-ripple, same pipeline")


## 10. Corner plot — full 3-way, once bilby's result is ready

Run section 9 above first (loads `g_arr`, `r_arr`, `meta`-equivalents from
Drive). Then run this once you've copied `bilby_bbh_case/bbh_case_result.json`
from the cluster to your Mac and uploaded *that one file* to Drive at
`MyDrive/mlgw_injection_benchmark/bbh_case/bbh_case_result.json`. Safe to
re-run repeatedly (e.g. after a fresh Drive upload).

In [ ]:
import json
from pathlib import Path

BILBY_RESULT = DRIVE_BASE / 'bbh_case_result.json'
if not BILBY_RESULT.is_file():
    raise FileNotFoundError(
        f"{BILBY_RESULT} not found. Copy bilby_bbh_case/bbh_case_result.json from the "
        "cluster to your Mac, then upload it to your Drive at this path.")

bres = json.load(open(BILBY_RESULT))
bpost_all = bres["posterior"]["content"]
def bcol(name):
    return np.asarray(bpost_all[name], dtype=float)

MAP_cmp = {
    "mc": ("chirp_mass", None), "q": ("mass_ratio", None),
    "chi_1": ("chi_1", None), "chi_2": ("chi_2", None),
    "distance": ("luminosity_distance", None), "inclination": ("theta_jn", None),
    "ra": ("ra", None), "dec": ("dec", None), "psi": ("psi", None),
    "phi_c": ("phase", None), "tc": ("geocent_time", lambda x: x - start_time),
}
b_arr = np.column_stack([
    MAP_cmp[n][1](bcol(MAP_cmp[n][0])) if MAP_cmp[n][1] else bcol(MAP_cmp[n][0]) for n in names_cmp])

lo3 = np.minimum.reduce([g_arr.min(axis=0), r_arr.min(axis=0), b_arr.min(axis=0)])
hi3 = np.maximum.reduce([g_arr.max(axis=0), r_arr.max(axis=0), b_arr.max(axis=0)])
pad3 = 0.03 * (hi3 - lo3)
ranges3 = list(zip(lo3 - pad3, hi3 + pad3))
ckw3 = dict(ckw); ckw3["range"] = ranges3

fig3 = corner.corner(b_arr, color="#3f4bd8", hist_kwargs=dict(density=True, color="#3f4bd8"), **ckw3)
corner.corner(g_arr, fig=fig3, color="#d8443f", hist_kwargs=dict(density=True, color="#d8443f"), **ckw3)
corner.corner(r_arr, fig=fig3, color="#3fa84b", hist_kwargs=dict(density=True, color="#3fa84b"), **ckw3)
fig3.legend(handles=[
    mlines.Line2D([], [], color="#3f4bd8", label="bilby + dynesty (SEOBNRv5_ROM)"),
    mlines.Line2D([], [], color="#d8443f", label="gwjax + acceptance-walk (mlgw_bbh_jax)"),
    mlines.Line2D([], [], color="#3fa84b", label="gwjax + acceptance-walk (ripplegw IMRPhenomXAS)"),
], loc="upper right", fontsize=14, frameon=False)
fig3.savefig("bbh_case_comparison_3way.png", dpi=150, bbox_inches="tight")
plt.show()

noise_logl = gres["log_noise_evidence"]
dbf = gres["logZ"] - noise_logl - bres["log_bayes_factor"]
sig_gb = (gres["logZ_err"]**2 + bres["log_evidence_err"]**2) ** 0.5
drb = rres["logZ"] - noise_logl - bres["log_bayes_factor"]
sig_rb = (rres["logZ_err"]**2 + bres["log_evidence_err"]**2) ** 0.5
print(f"bilby        logBF = {bres['log_bayes_factor']:.4f} +/- {bres['log_evidence_err']:.4f}")
print(f"gwjax        logBF = {gres['logZ'] - noise_logl:.4f} +/- {gres['logZ_err']:.4f}")
print(f"gwjax-ripple logBF = {rres['logZ'] - noise_logl:.4f} +/- {rres['logZ_err']:.4f}")
print(f"Delta logBF (gwjax - bilby)        = {dbf:+.4f}  ({dbf/sig_gb:+.1f} sigma)")
print(f"Delta logBF (gwjax-ripple - bilby)  = {drb:+.4f}  ({drb/sig_rb:+.1f} sigma) "
      "— pipeline+waveform-family systematic only, no surrogate")


## 11. Corner plot — bilby `IMRPhenomXAS` vs both gwjax runs

This bilby run finished first (faster per-likelihood-call than
`SEOBNRv5_ROM`) and is actually the **cleanest** comparison available:
ripplegw's `IMRPhenomXAS` and LAL's `IMRPhenomXAS` are independently known
to agree to ~1e-9 (`frozen_benchmark/README.md`), so **bilby (blue) vs
gwjax-ripple (green)** here isolates the sampler/pipeline difference with
essentially zero waveform-systematic contamination — unlike the
`SEOBNRv5_ROM`-vs-`mlgw_bbh_jax` comparison, which always carries the small
~0.6% surrogate effect. `gwjax-mlgw` (red) is included too for the full
picture. Run section 9 first (needs `g_arr`, `r_arr`, `start_time`-equivalents
from Drive) — section 10 is NOT required (this cell defines its own
bilby-column mapping). Upload `bilby_bbh_case_imrphenomxas/bbh_case_imrphenomxas_result.json`
to Drive at `MyDrive/mlgw_injection_benchmark/bbh_case/bbh_case_imrphenomxas_result.json`
first (already done if you're reading this after uploading it).

In [ ]:
import json
from pathlib import Path

BILBY_XAS_RESULT = DRIVE_BASE / 'bbh_case_imrphenomxas_result.json'
if not BILBY_XAS_RESULT.is_file():
    raise FileNotFoundError(
        f"{BILBY_XAS_RESULT} not found. Copy "
        "bilby_bbh_case_imrphenomxas/bbh_case_imrphenomxas_result.json from the "
        "cluster to your Mac, then upload it to your Drive at this path.")

bxas = json.load(open(BILBY_XAS_RESULT))
bxas_post_all = bxas["posterior"]["content"]
def bxas_col(name):
    return np.asarray(bxas_post_all[name], dtype=float)

# Self-contained (does NOT require section 10 to have run) — same map as
# section 10's MAP_cmp, gwjax name -> (bilby name, transform on bilby column).
MAP_cmp = {
    "mc": ("chirp_mass", None), "q": ("mass_ratio", None),
    "chi_1": ("chi_1", None), "chi_2": ("chi_2", None),
    "distance": ("luminosity_distance", None), "inclination": ("theta_jn", None),
    "ra": ("ra", None), "dec": ("dec", None), "psi": ("psi", None),
    "phi_c": ("phase", None), "tc": ("geocent_time", lambda x: x - start_time),
}

bxas_arr = np.column_stack([
    MAP_cmp[n][1](bxas_col(MAP_cmp[n][0])) if MAP_cmp[n][1] else bxas_col(MAP_cmp[n][0]) for n in names_cmp])

lo11 = np.minimum.reduce([g_arr.min(axis=0), r_arr.min(axis=0), bxas_arr.min(axis=0)])
hi11 = np.maximum.reduce([g_arr.max(axis=0), r_arr.max(axis=0), bxas_arr.max(axis=0)])
pad11 = 0.03 * (hi11 - lo11)
ranges11 = list(zip(lo11 - pad11, hi11 + pad11))
ckw11 = dict(ckw); ckw11["range"] = ranges11

fig11 = corner.corner(bxas_arr, color="#3f4bd8", hist_kwargs=dict(density=True, color="#3f4bd8"), **ckw11)
corner.corner(g_arr, fig=fig11, color="#d8443f", hist_kwargs=dict(density=True, color="#d8443f"), **ckw11)
corner.corner(r_arr, fig=fig11, color="#3fa84b", hist_kwargs=dict(density=True, color="#3fa84b"), **ckw11)
fig11.legend(handles=[
    mlines.Line2D([], [], color="#3f4bd8", label="bilby + dynesty (IMRPhenomXAS)"),
    mlines.Line2D([], [], color="#d8443f", label="gwjax + acceptance-walk (mlgw_bbh_jax)"),
    mlines.Line2D([], [], color="#3fa84b", label="gwjax + acceptance-walk (ripplegw IMRPhenomXAS)"),
], loc="upper right", fontsize=14, frameon=False)
fig11.savefig("bbh_case_comparison_imrphenomxas.png", dpi=150, bbox_inches="tight")
plt.show()

noise_logl = gres["log_noise_evidence"]
dbg = gres["logZ"] - noise_logl - bxas["log_bayes_factor"]
sig_bg = (gres["logZ_err"]**2 + bxas["log_evidence_err"]**2) ** 0.5
drbxas = rres["logZ"] - noise_logl - bxas["log_bayes_factor"]
sig_rbxas = (rres["logZ_err"]**2 + bxas["log_evidence_err"]**2) ** 0.5
print(f"bilby (IMRPhenomXAS) logBF = {bxas['log_bayes_factor']:.4f} +/- {bxas['log_evidence_err']:.4f}")
print(f"gwjax        logBF = {gres['logZ'] - noise_logl:.4f} +/- {gres['logZ_err']:.4f}")
print(f"gwjax-ripple logBF = {rres['logZ'] - noise_logl:.4f} +/- {rres['logZ_err']:.4f}")
print(f"Delta logBF (gwjax-ripple - bilby_IMRPhenomXAS) = {drbxas:+.4f}  ({drbxas/sig_rbxas:+.1f} sigma) "
      "— pipeline-only, same waveform family on both sides (the cleanest check)")
print(f"Delta logBF (gwjax-mlgw   - bilby_IMRPhenomXAS) = {dbg:+.4f}  ({dbg/sig_bg:+.1f} sigma) "
      "— surrogate + pipeline combined")


## 12. Corner plot — all four posteriors together

Everything in one overlay: bilby `SEOBNRv5_ROM` (blue), bilby `IMRPhenomXAS`
(purple), gwjax `mlgw_bbh_jax` (red), gwjax-ripple `IMRPhenomXAS` (green).
Requires sections 9, 10, **and** 11 to have run first in this session (reuses
`g_arr`, `r_arr`, `b_arr`, `bxas_arr` — no file I/O of its own).

In [ ]:
lo4 = np.minimum.reduce([g_arr.min(axis=0), r_arr.min(axis=0),
                         b_arr.min(axis=0), bxas_arr.min(axis=0)])
hi4 = np.maximum.reduce([g_arr.max(axis=0), r_arr.max(axis=0),
                         b_arr.max(axis=0), bxas_arr.max(axis=0)])
pad4 = 0.03 * (hi4 - lo4)
ranges4 = list(zip(lo4 - pad4, hi4 + pad4))
ckw4 = dict(ckw); ckw4["range"] = ranges4

fig4 = corner.corner(b_arr, color="#3f4bd8", hist_kwargs=dict(density=True, color="#3f4bd8"), **ckw4)
corner.corner(bxas_arr, fig=fig4, color="#8e44ad", hist_kwargs=dict(density=True, color="#8e44ad"), **ckw4)
corner.corner(g_arr, fig=fig4, color="#d8443f", hist_kwargs=dict(density=True, color="#d8443f"), **ckw4)
corner.corner(r_arr, fig=fig4, color="#3fa84b", hist_kwargs=dict(density=True, color="#3fa84b"), **ckw4)
fig4.legend(handles=[
    mlines.Line2D([], [], color="#3f4bd8", label="bilby + dynesty (SEOBNRv5_ROM)"),
    mlines.Line2D([], [], color="#8e44ad", label="bilby + dynesty (IMRPhenomXAS)"),
    mlines.Line2D([], [], color="#d8443f", label="gwjax + acceptance-walk (mlgw_bbh_jax)"),
    mlines.Line2D([], [], color="#3fa84b", label="gwjax + acceptance-walk (ripplegw IMRPhenomXAS)"),
], loc="upper right", fontsize=14, frameon=False)
fig4.savefig("bbh_case_comparison_4way.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"bilby SEOBNRv5_ROM   logBF = {bres['log_bayes_factor']:.4f} +/- {bres['log_evidence_err']:.4f}")
print(f"bilby IMRPhenomXAS   logBF = {bxas['log_bayes_factor']:.4f} +/- {bxas['log_evidence_err']:.4f}")
print(f"gwjax mlgw_bbh_jax   logBF = {gres['logZ'] - noise_logl:.4f} +/- {gres['logZ_err']:.4f}")
print(f"gwjax ripplegw       logBF = {rres['logZ'] - noise_logl:.4f} +/- {rres['logZ_err']:.4f}")


In [ ]:
from google.colab import runtime
runtime.unassign()


## Next steps

- **Right now**, section 9 above already gives you the gwjax-only corner
  plot (mlgw_bbh_jax vs ripplegw).
- On the IGWN/LDG cluster (or any CPU machine with bilby + lalsuite):
  `python run_bilby_bbh.py --npool 16` (see
  `examples/mlgw_injection_benchmark/bbh_case/README.md`).
- Once that finishes, copy `bilby_bbh_case/bbh_case_result.json` back to
  your Mac, upload it to Drive at
  `MyDrive/mlgw_injection_benchmark/bbh_case/bbh_case_result.json`, and
  run section 10 above for the full 3-way overlay.
- Locally instead (equivalent, from the command line once both results are
  on your Mac):
  ```
  python make_comparison_plot.py \
      --gwjax gwjax_bbh_case \
      --bilby bilby_bbh_case/bbh_case_result.json \
      --gwjax-ripple gwjax_bbh_case_ripple \
      --out   bbh_case_comparison.png
  ```
  (`--bilby` and `--gwjax-ripple` are both optional there too.)
